<a href="https://colab.research.google.com/github/kavyajeetbora/foursquare_ai/blob/master/notebooks/16%20-%20Generate%20tiles%20db%20from%20overture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install --upgrade duckdb

In [1]:
import duckdb
print(duckdb.__version__)          # should be 1.1.x or newer in 2026

con = duckdb.connect()           # or duckdb.connect(':memory:')

con.sql("INSTALL spatial;")
con.sql("INSTALL httpfs;")       # almost always needed for s3:// paths

con.sql("LOAD spatial;")
con.sql("LOAD httpfs;")

1.4.4


In [ ]:
con = duckdb.connect('tiles.db')  # or choose a new name e.g. 'nyc_buildings.db' to avoid confusion

con.sql("INSTALL spatial; LOAD spatial;")   # make sure extension is ready

con.sql("""
CREATE OR REPLACE TABLE t1 AS (
  SELECT
    names.primary AS "name",
    addresses[1].country AS country,
    ST_Transform(geometry, 'EPSG:4326', 'EPSG:3857', always_xy := true) AS geometry
  FROM read_parquet('s3://overturemaps-us-west-2/release/2026-02-18.0/theme=places/*/*')
  WHERE 
    country = 'IN' AND categories.primary ilike '%restaurant%'
);
""")

con.sql("CREATE INDEX my_idx ON t1 USING RTREE (geometry);")

# Quick reality check — should print a number > 0 if data loaded
con.sql("SELECT COUNT(*) FROM t1").show()

# Optional: force write to disk and close cleanly
con.sql("CHECKPOINT")
con.close()

In [ ]:
!ls -lh tiles.db

In [2]:
# 1. Basic smoke test – does DuckDB even see the file as non-empty?
con = duckdb.connect(r'..\data\tiles.db')
print(con.sql("SELECT * FROM duckdb_tables()").fetchall())          # lists user tables
print(con.sql("SELECT * FROM duckdb_databases()").fetchall())       # should show main db
print(con.sql("PRAGMA database_size").fetchall())                   # rough size info

# 2. Also check for any leftover attached databases
con.sql("SHOW DATABASES").show()

[('tiles', 609, 'main', 590, 't1', 593, None, {}, False, False, False, 190912, 3, 1, 0, 'CREATE TABLE t1("name" VARCHAR, country VARCHAR, geometry GEOMETRY);')]
[('tiles', 609, 'c:\\users\\kavya\\downloads\\projects\\nyc_building_explorer\\data-engineering\\..\\data\\tiles.db', None, {'storage_version': 'v1.0.0+'}, False, 'duckdb', False, False, None), ('system', 0, None, None, {}, True, 'duckdb', False, False, None), ('temp', 2010, None, None, {}, True, 'duckdb', False, False, None)]
[('tiles', '10.5 MiB', 262144, 42, 42, 0, '0 bytes', '256.0 KiB', '12.5 GiB')]
┌───────────────┐
│ database_name │
│    varchar    │
├───────────────┤
│ tiles         │
└───────────────┘



In [6]:
con = duckdb.connect(r'..\data\tiles.db')   # creates file in Colab's /content/
# then run the same INSTALL / LOAD commands
# then your original code with attach/use/detach

con.sql("SELECT COUNT(*) FROM t1").show()

con.sql("""
SELECT
  COUNT(*) AS cnt,
FROM t1
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       190912 │
└──────────────┘

┌────────┐
│  cnt   │
│ int64  │
├────────┤
│ 190912 │
└────────┘

